### Imports

In [26]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval import evaluate, metrics
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
import pandas as pd
from deepeval.models import OllamaModel
from pathlib import Path


# Walk up from notebook dir until we find the project root containing .env.local
_dir = Path.cwd()
while not (_dir / ".env.local").exists() and _dir != _dir.parent:
    _dir = _dir.parent
env_path = _dir / ".env.local"
print("Using:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

Using: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


### Initialise the Judge

In [28]:
import re
from deepeval.models import OllamaModel

class CleanOllamaModel(OllamaModel):
    """OllamaModel that strips markdown code blocks from LLM responses."""
    
    @staticmethod
    def _strip_markdown_json(text: str) -> str:
        return re.sub(r'^```(?:json)?\s*\n?', '', text.strip()).rstrip('`').strip()

    def generate(self, prompt, schema=None):
        result, cost = super().generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

    async def a_generate(self, prompt, schema=None):
        result, cost = await super().a_generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

In [29]:
judgeModel = CleanOllamaModel(
    model="qwen3-next:80b-cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},   
)

In [30]:
print(judgeModel.generate("What is the capital of Italy?"))

('The capital of Italy is **Rome** (Italian: *Roma*).  \n\nIt has been the capital since 1871, following the unification of Italy (*Risorgimento*). Rome is not only the political center of the country but also its largest city and a global hub for history, culture, art, and religion (home to the Vatican City, an independent city-state within Rome). 🌍🇮🇹', 0)


### Initialise the Candidate

In [31]:
candidateModel = ChatOllama(
    base_url=CLOUD_MODEL_BASE_URL,
    model="kimi-k2.6:cloud",
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},  
)

In [32]:
print(candidateModel.invoke("What is the capital of Italy?").content)

The capital of Italy is **Rome** (Roma in Italian).


###  Creating Test Data for Goldens/LLMTestCases

In [33]:
test_data = [
  {
    "input": "Antworte in 2 Sätzen: Warum empfiehlt die TestGilde ausdrücklich, mit der Automatisierung im Quadranten A3 zu starten – und nicht in A1 oder A2?",
    "expected_output": "Die TestGilde empfiehlt den Start im Quadranten A3, weil dort Testobjekte mit hohem Risiko, aber geringer Komplexität liegen – somit lassen sie sich schnell automatisieren und häufig ausführen, was eine schnelle Amortisation ermöglicht."
  },
  {
    "input": "Antworte in 2 Sätzen: Welche drei technischen Faktoren nennt die TestGilde, die den Erfolg einer Testautomation entscheidend beeinflussen?",
    "expected_output": "Die drei technischen Erfolgsfaktoren laut TestGilde sind: (1) die Robustheit des Zusammenspiels von Anwendung und Testroboter, (2) die Qualität der GUI-Element-Attribute (z. B. keine dynamischen IDs) und (3) die Rücksetzbarkeit der Testdaten."
  },
  {
    "input": "Antworte in 2 Sätzen: Nach TestGilde: Warum sollte die Ausdehnung der Testautomation über Quadrant A3 hinaus nicht ohne weitere Prüfung erfolgen?",
    "expected_output": "Laut TestGilde sollte die Ausdehnung über Quadrant A3 hinaus nur begrenzt erfolgen, weil mit steigender Komplexität und/oder sinkendem Risiko der Aufwand für Automatisierung und Wartung schneller steigt als der Nutzen durch häufige Ausführung – ab da spricht man von Verschwendung."
  }
]

### Convert Test Data into Goldens Data Set(Ground Truth)

In [34]:
goldens = [Golden(input=d["input"], expected_output=d["expected_output"]) for d in test_data]
dataset = EvaluationDataset(goldens=goldens)


In [35]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Antworte in 2 Sätzen: Warum empfiehlt die TestGilde ausdrücklich, mit der Automatisierung im Quadranten A3 zu starten – und nicht in A1 oder A2?', actual_output=None, expected_output='Die TestGilde empfiehlt den Start im Quadranten A3, weil dort Testobjekte mit hohem Risiko, aber geringer Komplexität liegen – somit lassen sie sich schnell automatisieren und häufig ausführen, was eine schnelle Amortisation ermöglicht.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Antworte in 2 Sätzen: Welche drei technischen Faktoren nennt die TestGilde, die den Erfolg einer Testautomation entscheidend beeinflussen?', actual_output=None, expected_output='Die drei technischen Erfolgsfaktoren laut TestGilde sind: (1) die Robustheit des Zusammenspiels von Anwendung und Testroboter, (2) die Qualität der G

### Optional: Push the Goldens Data Set to Confident AI (for reuse with different LLMs)

In [7]:
dataset.push("Answer Relevancy Dataset – Test")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=169902;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpbnfca9000smy13nv51byl9\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpbnfca9000smy13nv51byl9]8;;\

### Optional: Pull the current dataset and convert it into LLMTestCases
##### This is only needed if you already have a dataset in Confident AI and want to evalute different LLMs with it

In [117]:
#dataset.pull(alias="Answer Relevancy Dataset", auto_convert_goldens_to_test_cases=True) 

### Add actual output from the candidate LLM to the Goldens Data Set

In [36]:
for g in dataset.goldens:
    response = candidateModel.invoke(g.input)
    g.actual_output = getattr(response, "content", str(response))


In [37]:
print(dataset)

EvaluationDataset(test_cases=[], goldens=[Golden(input='Antworte in 2 Sätzen: Warum empfiehlt die TestGilde ausdrücklich, mit der Automatisierung im Quadranten A3 zu starten – und nicht in A1 oder A2?', actual_output='Die TestGilde empfiehlt den Einstieg in die Automatisierung im Quadranten A3, weil dieser Bereich – zumeist die API- oder Integrationsebene – die optimale Balance aus fachlicher Relevanz und technischer Stabilität bietet, während A1 (Unit-Ebene) klassischerweise in der Verantwortung der Entwickler liegt und A2 (GUI-Ebene) durch häufige Oberflächenänderungen zu hohen Wartungskosten und instabilen Tests führt. Dadurch lassen sich in A3 automatisierte Tests schneller implementieren, zuverlässiger über längere Zeit betreiben und mit einem deutlich besseren Return on Investment skalieren als in den anderen beiden Quadranten.', expected_output='Die TestGilde empfiehlt den Start im Quadranten A3, weil dort Testobjekte mit hohem Risiko, aber geringer Komplexität liegen – somit la

#### Convert Goldens to LLMTestCases for Evaluation via DeepEval

In [38]:
test_cases = [
    LLMTestCase(
        input=g.input,
        expected_output=g.expected_output,
        actual_output=getattr(g, "actual_output", None)
    )
    for g in dataset.goldens
]

In [39]:
test_cases

[LLMTestCase(input='Antworte in 2 Sätzen: Warum empfiehlt die TestGilde ausdrücklich, mit der Automatisierung im Quadranten A3 zu starten – und nicht in A1 oder A2?', actual_output='Die TestGilde empfiehlt den Einstieg in die Automatisierung im Quadranten A3, weil dieser Bereich – zumeist die API- oder Integrationsebene – die optimale Balance aus fachlicher Relevanz und technischer Stabilität bietet, während A1 (Unit-Ebene) klassischerweise in der Verantwortung der Entwickler liegt und A2 (GUI-Ebene) durch häufige Oberflächenänderungen zu hohen Wartungskosten und instabilen Tests führt. Dadurch lassen sich in A3 automatisierte Tests schneller implementieren, zuverlässiger über längere Zeit betreiben und mit einem deutlich besseren Return on Investment skalieren als in den anderen beiden Quadranten.', expected_output='Die TestGilde empfiehlt den Start im Quadranten A3, weil dort Testobjekte mit hohem Risiko, aber geringer Komplexität liegen – somit lassen sie sich schnell automatisieren

### Define metric

##### The answer relevancy metric first breaks down the actual output of a test case into distinct statements, then calculates the proportion of those statements that are relevant to the given input.

$ \text{Answer Relevancy} = \frac{\text{Number of relevant statements in actual output}}{\text{Total number of statements in actual output}} $

##### The final score is the proportion of relevant statements found in the actual output.

#### How to interpret the Answer Relevancy Score

| **Score Range** | **Meaning** | **Example Behavior** |
|------------------|-------------|-----------------------|
| **0.8 → 1.0** | 🟢 **Highly relevant** | Fully answers the question with no unnecessary or off-topic information |
| **0.6 → 0.8** | 🟡 **Mostly relevant** | Correct but may include minor irrelevant or redundant details |
| **0.3 → 0.6** | 🟠 **Partially relevant** | Addresses only part of the question or mixes in unrelated content |
| **0.0 → 0.3** | 🔴 **Irrelevant / off-topic** | Fails to answer or completely diverges from the question |

In [40]:
metric = AnswerRelevancyMetric(model=judgeModel)

### Execute evaluation with DeepEval and ConfidentAI

In [41]:
results = evaluate(test_cases=test_cases, metrics=[metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3-next:80b-cloud (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3-next:80b-cloud (Ollama), reason: The score is 1.00 because the response is fully relevant with no irrelevant statements., error: None)

For test case:

  - input: Antworte in 2 Sätzen: Welche drei technischen Faktoren nennt die TestGilde, die den Erfolg einer Testautomation entscheidend beeinflussen?
  - actual output: Die TestGilde identifiziert die Testbarkeit der Anwendung, die Stabilität und Wartbarkeit der Testframeworks sowie die Verfügbarkeit valider Testdaten als die drei zentralen technischen Erfolgsfaktoren. Diese Elemente bilden zusammen die technische Grundlage, ohne die eine skalierbare und langfristig erfolgreiche Testautomation nicht realisierbar ist.
  - expected output: Die drei technischen Erfolgsfaktoren laut TestGilde sind: (1) die Robustheit des Zusammenspiels von Anwendung und Testroboter, (2) die Qualität der GUI-Element-Attribute (z. B. keine dynamischen

⚠ WARNING: No hyperparameters logged.
» ]8;id=630889;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=743525;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpczea2j0008qt13u4mg33ee/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpczea2j0008qt13u4mg33ee/test-cases]8;;\

#### Display results in a pandas dataframe

In [42]:
rows = []
for tr in results.test_results:
    for m in tr.metrics_data:
        rows.append({
            "test_case": tr.name,
            "input": tr.input,
            "expected_output": tr.expected_output,
            "actual_output": tr.actual_output,
            "metric": m.name,
            "score": m.score,
            "threshold": m.threshold,
            "success": m.success,
            "reason": m.reason,
            "evaluation_model": m.evaluation_model,
            "evaluation_cost": m.evaluation_cost,
        })

results_df = pd.DataFrame(rows)
display(results_df)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
0,test_case_1,Antworte in 2 Sätzen: Welche drei technischen ...,Die drei technischen Erfolgsfaktoren laut Test...,Die TestGilde identifiziert die Testbarkeit de...,Answer Relevancy,1.0,0.5,True,The score is 1.00 because the response is full...,qwen3-next:80b-cloud (Ollama),0.0
1,test_case_2,Antworte in 2 Sätzen: Nach TestGilde: Warum so...,Laut TestGilde sollte die Ausdehnung über Quad...,Eine Ausdehnung der Testautomation über Quadra...,Answer Relevancy,0.5,0.5,True,The score is 0.50 because the response incorre...,qwen3-next:80b-cloud (Ollama),0.0
2,test_case_0,Antworte in 2 Sätzen: Warum empfiehlt die Test...,Die TestGilde empfiehlt den Start im Quadrante...,Die TestGilde empfiehlt den Einstieg in die Au...,Answer Relevancy,1.0,0.5,True,The score is 1.00 because all parts of the res...,qwen3-next:80b-cloud (Ollama),0.0


#### Evaluate the results and add a suggestion for improvements

In [17]:
with pd.option_context("display.max_colwidth", None):
    failing = results_df[results_df["success"].astype(str).str.lower().eq("false")]
    display(failing)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
